# Symbolic AI Concepts (Search Mini-Notebook)
## Heuristic Search: BFS, DFS, A\* — and why it still matters in the LLM era

Modern LLM systems often *look* like they “reason”, but under the hood many strong systems rely on **search**:
- **Planning**: explore action sequences to reach a goal (tool-use, web actions, API calls).
- **Verification**: search for counterexamples, constraint violations, or alternative proofs.
- **Reliability**: when a single-shot answer is brittle, search provides **backtracking** and **best-first exploration**.

In this notebook you will:
- implement **BFS**, **DFS**, and **A\*** (minimal, readable Python),
- run them on a small **grid world**,
- compare **path quality** and **nodes expanded**,
- see how a **heuristic** changes the game,
- and connect these ideas to **LLM agents** (planner–executor loops, tree search, ToT-style exploration).

> **No external dependencies** required (pure Python). An optional note at the end lists useful libraries.


## Agenda
1. Search framing: states, edges, costs, goals
2. Problem setup: a grid world with obstacles
3. BFS (optimal for unweighted graphs)
4. DFS (fast to find *a* solution, not necessarily good)
5. A\* (best-first with heuristics; optimal with admissible heuristic)
6. Why search matters for LLM agents
7. Practical solver/library recommendations


---
## 0) Search as a universal abstraction

We’ll model problems as:
- **State**: what the world looks like right now (e.g., agent position, tool outputs, memory)
- **Actions / transitions**: how you can move to a new state
- **Goal test**: how you know you’re done
- **Cost**: optionally, how expensive actions are

Search algorithms differ mainly in *which frontier node they expand next*:
- **BFS**: expands by increasing depth (shortest number of steps in unweighted graphs)
- **DFS**: goes deep first (low memory, but can be unlucky)
- **A\***: expands by increasing `f(n)=g(n)+h(n)`  
  - `g(n)` = cost so far  
  - `h(n)` = heuristic estimate of remaining cost  
  - With an *admissible* `h`, A\* is optimal and often vastly more efficient than BFS.


---
## 1) Grid world problem

We’ll plan from **Start** to **Goal** in a 2D grid with obstacles.
Moves are 4-connected (up/down/left/right), each with cost 1.

This toy example maps nicely to LLM agents:
- a “state” could be the **current plan prefix** or **current environment snapshot**
- an “action” could be **calling a tool**, **clicking a UI element**, **asking a sub-question**
- the goal could be a **verified final answer** or **successful task completion**


In [ ]:
from collections import deque
import heapq
import random

def make_grid(width, height, walls):
    # Return a set of blocked cells (x,y).
    return set(walls)

def in_bounds(x, y, width, height):
    return 0 <= x < width and 0 <= y < height

def neighbors(pos, blocked, width, height):
    x, y = pos
    for dx, dy in [(1,0), (-1,0), (0,1), (0,-1)]:
        nx, ny = x+dx, y+dy
        if in_bounds(nx, ny, width, height) and (nx, ny) not in blocked:
            yield (nx, ny)

def reconstruct_path(parent, start, goal):
    if start == goal:
        return [start]
    if goal not in parent:
        return None
    cur = goal
    path = [cur]
    while cur != start:
        cur = parent[cur]
        path.append(cur)
    path.reverse()
    return path

def render_grid(width, height, blocked, start, goal, path=None):
    path = set(path or [])
    lines = []
    for y in range(height):
        row = []
        for x in range(width):
            p = (x, y)
            if p == start:
                row.append("S")
            elif p == goal:
                row.append("G")
            elif p in blocked:
                row.append("#")
            elif p in path:
                row.append("·")
            else:
                row.append(" ")
        lines.append("".join(row))
    return "\n".join(lines)

# A small maze-like grid
W, H = 16, 9
walls = [
    (2,1),(3,1),(4,1),(5,1),(6,1),
    (6,2),(6,3),(6,4),(6,5),
    (9,0),(9,1),(9,2),(9,3),(9,4),(9,5),
    (12,3),(13,3),(14,3),
    (2,6),(3,6),(4,6),(5,6),(6,6),(7,6),(8,6),
    (12,6),(12,7),(12,8),
]
blocked = make_grid(W, H, walls)
start = (0, 0)
goal  = (15, 8)

print(render_grid(W, H, blocked, start, goal))


---
## 2) BFS (Breadth-First Search)

**Idea:** explore all nodes at depth 0, then depth 1, then depth 2, …  
**Guarantee (unweighted graphs):** finds a shortest path (fewest steps).  
**Cost:** can expand many nodes when the state space is large.

We’ll track:
- `expanded`: how many states we popped from the frontier and expanded
- `frontier_max`: peak frontier size (rough proxy for memory usage)


In [ ]:
def bfs(start, goal, blocked, width, height):
    q = deque([start])
    parent = {start: None}
    expanded = 0
    frontier_max = 1

    while q:
        frontier_max = max(frontier_max, len(q))
        cur = q.popleft()
        expanded += 1

        if cur == goal:
            return reconstruct_path(parent, start, goal), expanded, frontier_max

        for nxt in neighbors(cur, blocked, width, height):
            if nxt not in parent:
                parent[nxt] = cur
                q.append(nxt)

    return None, expanded, frontier_max

path_bfs, expanded_bfs, fmax_bfs = bfs(start, goal, blocked, W, H)
print("BFS path length:", None if path_bfs is None else len(path_bfs)-1)
print("BFS states expanded:", expanded_bfs)
print("BFS peak frontier size:", fmax_bfs)
print()
print(render_grid(W, H, blocked, start, goal, path_bfs))


---
## 3) DFS (Depth-First Search)

**Idea:** go as deep as possible before backtracking.  
**Pros:** very low memory, can find *a* solution quickly on some problems.  
**Cons:** not optimal, and can get unlucky (explore long dead-ends first).

We’ll implement an iterative DFS with an explicit stack to keep it simple and safe.


In [ ]:
def dfs(start, goal, blocked, width, height):
    stack = [start]
    parent = {start: None}
    expanded = 0
    frontier_max = 1

    while stack:
        frontier_max = max(frontier_max, len(stack))
        cur = stack.pop()
        expanded += 1

        if cur == goal:
            return reconstruct_path(parent, start, goal), expanded, frontier_max

        # Neighbor ordering affects DFS a lot; we keep it deterministic.
        nxts = list(neighbors(cur, blocked, width, height))
        for nxt in reversed(nxts):  # LIFO: push in reverse to explore "first" earlier
            if nxt not in parent:
                parent[nxt] = cur
                stack.append(nxt)

    return None, expanded, frontier_max

path_dfs, expanded_dfs, fmax_dfs = dfs(start, goal, blocked, W, H)
print("DFS path length:", None if path_dfs is None else len(path_dfs)-1)
print("DFS states expanded:", expanded_dfs)
print("DFS peak frontier size:", fmax_dfs)
print()
print(render_grid(W, H, blocked, start, goal, path_dfs))


---
## 4) A\* Search

A\* uses **best-first** expansion based on:

\[
f(n) = g(n) + h(n)
\]

- `g(n)`: exact cost from start to `n` (in our grid, number of steps so far)
- `h(n)`: heuristic estimate from `n` to goal

A standard heuristic here is **Manhattan distance**:

\[
h(x,y) = |x-x_g| + |y-y_g|
\]

Key fact:
- Manhattan distance is **admissible** for 4-connected unit-cost grids (never overestimates),
  so A\* is **optimal** like BFS but often explores far fewer states.


In [ ]:
def manhattan(a, b):
    return abs(a[0]-b[0]) + abs(a[1]-b[1])

def astar(start, goal, blocked, width, height, h_fn=manhattan):
    # priority queue holds (f, g, state)
    pq = []
    heapq.heappush(pq, (h_fn(start, goal), 0, start))

    parent = {start: None}
    gbest = {start: 0}

    expanded = 0
    frontier_max = 1

    while pq:
        frontier_max = max(frontier_max, len(pq))
        f, g, cur = heapq.heappop(pq)
        expanded += 1

        if cur == goal:
            return reconstruct_path(parent, start, goal), expanded, frontier_max

        for nxt in neighbors(cur, blocked, width, height):
            cand_g = g + 1
            if nxt not in gbest or cand_g < gbest[nxt]:
                gbest[nxt] = cand_g
                parent[nxt] = cur
                cand_f = cand_g + h_fn(nxt, goal)
                heapq.heappush(pq, (cand_f, cand_g, nxt))

    return None, expanded, frontier_max

path_astar, expanded_astar, fmax_astar = astar(start, goal, blocked, W, H)
print("A* path length:", None if path_astar is None else len(path_astar)-1)
print("A* states expanded:", expanded_astar)
print("A* peak frontier size:", fmax_astar)
print()
print(render_grid(W, H, blocked, start, goal, path_astar))


---
## 5) Side-by-side comparison

You should notice:
- **BFS** and **A\*** give the same shortest-path length here (both optimal in this setting).
- **DFS** can find a longer path (or sometimes a short one if lucky).
- **A\*** usually expands fewer states than BFS thanks to the heuristic.


In [ ]:
def summarize(name, path, expanded, fmax):
    length = None if path is None else len(path) - 1
    return {"algo": name, "path_len": length, "expanded": expanded, "peak_frontier": fmax}

table = [
    summarize("BFS", path_bfs, expanded_bfs, fmax_bfs),
    summarize("DFS", path_dfs, expanded_dfs, fmax_dfs),
    summarize("A*",  path_astar, expanded_astar, fmax_astar),
]
table


---
## 6) Heuristics as “guidance” (LLM analogy)

In A\*, the heuristic `h(n)` is the *guidance signal*.  
In many LLM-era agent systems, the LLM plays a similar role:

- proposes **which branch** to explore (next subgoal / action / tool call),
- estimates “how close” we are to the goal,
- ranks candidates.

But LLM guidance is imperfect. Search still matters for:
- **backtracking** from wrong steps,
- exploring alternatives,
- enforcing **constraints** (schemas, safety rules, type checks),
- and selecting the best candidate via explicit scoring.

Below we compare:
- A\* with a good heuristic (Manhattan distance)
- A\* with a **noisy heuristic** (pretend we have a slightly unreliable guide)


In [ ]:
def noisy_manhattan(a, b, noise_scale=3.0, seed=42):
    # Deterministic noise per state for reproducibility
    rnd = random.Random(hash((a, b, seed)) & 0xffffffff)
    return manhattan(a, b) + rnd.uniform(-noise_scale, noise_scale)

def h_noisy(a, b):
    return noisy_manhattan(a, b, noise_scale=3.0, seed=42)

path_good, ex_good, _ = astar(start, goal, blocked, W, H, h_fn=manhattan)
path_noisy, ex_noisy, _ = astar(start, goal, blocked, W, H, h_fn=h_noisy)

print("A* (good h)  path_len:", len(path_good)-1, "expanded:", ex_good)
print("A* (noisy h) path_len:", len(path_noisy)-1, "expanded:", ex_noisy)


### Interpretation

- You’ll often see **more expansions** with the noisy heuristic.
- This is exactly why LLM agents benefit from explicit search + bookkeeping:
  even a strong model will sometimes “choose the wrong branch” early.

If you only do one mental upgrade for agentic systems:
> **Treat reasoning as search over structured states**, with the model providing guidance—not guarantees.


---
## 7) Why search is important in the LLM era (practical)

**Planner–Executor–Tool architectures** often look like:
1. Expand candidate plans / next actions (LLM proposes)
2. Score / verify / simulate (tools + constraints)
3. Select best and continue (search control)

Where BFS/DFS/A\* show up:
- **BFS**: explore shallow solutions first (e.g., minimal tool calls)
- **DFS**: commit to a plan, but backtrack on failure (common in scripted agents)
- **A\*/best-first**: prioritize promising partial plans using a heuristic:
  - learned value models,
  - LLM self-evaluation,
  - constraint satisfaction progress,
  - or domain heuristics.

Search gives you:
- **robustness** (recovery and backtracking),
- **traceability** (why a path/plan was chosen),
- **control** (limits on branching, cost, risk),
- **and correctness hooks** (constraints, verifiers).


---
## 8) Recommended libraries / solvers (when you don’t want to re-implement)

**Graph search / shortest path**
- `networkx` — excellent graph algorithms (BFS/DFS/shortest_path)
- Keep using `heapq` (like above) for custom A\* in production code (common approach)

**Planning (PDDL-style, beyond this notebook)**
- `pyperplan` — lightweight classical planner (good for learning and experiments)
- `unified-planning` — richer planning interface with multiple backends

**Constraint optimization (often paired with planning)**
- Google `ortools` — CP-SAT solver, routing, scheduling

Rule of thumb:
- For understanding: implement BFS/DFS/A\* once.
- For real systems: use libraries and spend your energy on **state modeling + heuristics + constraints**.
